[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pmarcelino/mobillity-courses/blob/main/mobillity-univ/module-5-evaluating/notebook-5.4-auditing-a-complete-analysis.ipynb)


# Auditing an analysis you didn't write — is FGC's schedule as frequent as it looks?

**The question.** On a normal weekday, how often do the trains actually run on Barcelona's FGC network — the average number of departures per hour on each line, and which line runs most often? The phrase to pin down is *departures per hour*: a line that runs 180 times midnight and 5 a.m. is a very different service from one that runs 180 times spread across a full 24 hours. In this notebook, we'll ask the assisant to write the analysis/code and we'll audit it.

**Why it's worth asking.** Frequency is what a rider feels. It decides whether you check a timetable or just turn up at the platform, and it's the headline number a transit team would put in a service report. If it's off, every decision resting on it is off — and here the number comes from code an assistant wrote, not code we wrote ourselves. Accordingly, it is important to audit the analysis/code that produces to be sure it is correct.

**The data.** The FGC feed in GTFS — the standard public-transport schedule format used by transit apps: a table of *lines*, a table of *trips*, and a large table of *stop times* (one row for every stop of every trip, with the scheduled departure time), plus a calendar of which service runs on which date. Two important caveats to keep in mind: is the *scheduled* service for one representative weekday, not what actually ran; and the departure times follow the transit convention where a train leaving at a quarter past midnight is written "24:15", past the ordinary 24-hour clock.

**The method.** We treat the assistant's analysis as a draft to be *audited*, not trusted: run it, read its logic, spot-check a number, sanity-check the result, then have the assistant write checks and review its own work — and finish with a short table of what we found and fixed.

## 1. Ask the assistant to write the analysis

Most analyses now start here: describe the question and the data, and ask the assistant to write the code. We are directing it to *write the Python* rather than writing it ourselves — which is exactly why the checking habits that follow matter so much.

A prompt for this is:

```markdown
I have the FGC public-transport schedule in the standard GTFS format, as CSV text tables published online — each file sits at its own URL under a shared base address: `routes` (one row per line, with a `route_type` mode code), `trips` (one row per scheduled trip, linked to a line and to a service calendar), `stop_times` (one row per stop of every trip, with a `departure_time`), and `calendar_dates` (which service runs on which date).

Write Python with pandas that answers: on a representative weekday, what is the average number of departures per hour for each line? Load the tables from those URLs, keep only the trips that run on one normal weekday, count each line's departures, convert the departure times so they can be worked with, and turn each line's daily departures into an average per hour. Print a table of lines sorted from most to least frequent.
```

Here is the code it wrote:

In [1]:
"""
Average departures per hour for each FGC line, on a representative weekday.

Data: the FGC GTFS feed (routes, trips, stop_times, calendar_dates).
Columns used: routes.route_type, trips.service_id / route_id,
              stop_times.departure_time, calendar_dates.date / exception_type.
"""
import pandas as pd

# 0. Settings you can change
DATA_DIR = ("https://raw.githubusercontent.com/pmarcelino/mobillity-courses/main/"
            "mobillity-univ/module-5-evaluating/data/gtfs-catalonia-fgc")  # the FGC feed, hosted on GitHub (frozen snapshot)
SERVICE_DAY = "20260414"   # one normal weekday (YYYYMMDD)
TRAIN_MODE = 3             # which route_type counts as "the trains"

# 1. Load the tables (codes as text so nothing is silently converted)
routes = pd.read_csv(f"{DATA_DIR}/routes.txt", dtype=str)
trips = pd.read_csv(f"{DATA_DIR}/trips.txt", dtype=str)
stop_times = pd.read_csv(f"{DATA_DIR}/stop_times.txt", dtype=str)
calendar = pd.read_csv(f"{DATA_DIR}/calendar_dates.txt", dtype=str)
routes["route_type"] = routes["route_type"].astype(int)

# 2. Keep the trips that run on the chosen weekday, on the train lines
running = calendar[(calendar["date"] == SERVICE_DAY)
                   & (calendar["exception_type"] == "1")]
day = trips[trips["service_id"].isin(running["service_id"])]
day = day.merge(routes, on="route_id")
day = day[day["route_type"] == TRAIN_MODE]

# 3. Turn the departure clock times into real timestamps
stop_times["departure"] = pd.to_datetime(stop_times["departure_time"],
                                         format="%H:%M:%S")

# 4. Count each line's departures and average over the 24-hour day
per_line = day.groupby("route_short_name").size()
frequency = (per_line / 24).round(2).sort_values(ascending=False)
print("Departures per hour by line:")
print(frequency)

ValueError: time data "24:01:00" doesn't match format "%H:%M:%S". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

It didn't even finish. The last line is a `ValueError` about the time `"24:01:00"`. That's a plain crash, not a subtle logic slip — and a crash is the quickest kind of problem to hand straight back to the assistant.

## 2. Feed the error back and fix the time parsing

When the code throws, the fastest move is to paste the exact error back and ask for both an explanation and a corrected version — and, since we'll want to re-run this analysis a few times during the audit, to ask for it in a shape that's easy to adjust.

```markdown
I ran your code and got this error:

`ValueError: time data "24:01:00" doesn't match format "%H:%M:%S"`

Explain in plain terms why this happens, and give me corrected code. I'd like the fix to keep the analysis easy to re-run with a different set of lines later.
```

The assistant explains, then returns corrected code:

```markdown
**Why it happens.** GTFS uses a special clock. A trip that leaves just after midnight but belongs to the *previous* day's service is written past 24:00 — so "24:01" means one minute after midnight, and you'll even see times like "25:30". Those aren't valid on a normal 24-hour clock, so parsing them as clock times fails on the very first one.

**The fix.** Don't treat the value as a clock time at all — read it as *minutes since midnight* (hours × 60 + minutes). Then "24:01" is simply 1441 minutes, no error, and the after-midnight trips are still counted. I've also moved the line filter into a parameter so you can re-run the analysis with a different set of modes without editing the body.
```

In [2]:
"""
Departures per hour by line — corrected time handling, filter kept adjustable.

Fix: GTFS writes after-midnight departures past 24:00 (e.g. "24:15" = 00:15),
so we read each time as MINUTES SINCE MIDNIGHT instead of a 24-hour clock.
"""
import pandas as pd

# 0. Settings you can change
DATA_DIR = ("https://raw.githubusercontent.com/pmarcelino/mobillity-courses/main/"
            "mobillity-univ/module-5-evaluating/data/gtfs-catalonia-fgc")
SERVICE_DAY = "20260414"

def departures_per_hour(mode_codes, over_service_window=False):
    """Average departures per hour for each line running on SERVICE_DAY.
    mode_codes          : which route_type values to keep.
    over_service_window : if True, divide by each line's own operating hours
                          instead of by a flat 24.
    """
    routes = pd.read_csv(f"{DATA_DIR}/routes.txt", dtype=str)
    trips = pd.read_csv(f"{DATA_DIR}/trips.txt", dtype=str)
    stop_times = pd.read_csv(f"{DATA_DIR}/stop_times.txt", dtype=str)
    calendar = pd.read_csv(f"{DATA_DIR}/calendar_dates.txt", dtype=str)
    routes["route_type"] = routes["route_type"].astype(int)
    stop_times["stop_sequence"] = stop_times["stop_sequence"].astype(int)

    # 1. minutes since midnight (handles "24:15", "25:30", ...)
    hm = stop_times["departure_time"].str.split(":")
    stop_times["dep_min"] = hm.str[0].astype(int) * 60 + hm.str[1].astype(int)

    # 2. trips on the weekday, on the chosen modes
    running = calendar[(calendar["date"] == SERVICE_DAY)
                       & (calendar["exception_type"] == "1")]
    day = trips[trips["service_id"].isin(running["service_id"])].merge(
        routes[["route_id", "route_short_name", "route_type"]], on="route_id")
    day = day[day["route_type"].isin(mode_codes)]
    if day.empty:
        return pd.DataFrame(columns=["line", "trips", "per_hour"])

    # 3. each trip's own departure = the time at its first stop
    first_dep = (stop_times[stop_times["trip_id"].isin(day["trip_id"])]
                 .sort_values("stop_sequence")
                 .groupby("trip_id")["dep_min"].first())
    day = day.merge(first_dep.rename("dep_min"), on="trip_id")

    # 4. per line: count, service window, departures per hour
    g = day.groupby("route_short_name")["dep_min"]
    tbl = g.agg(trips="size", first_dep="min", last_dep="max").reset_index()
    tbl["service_h"] = (tbl["last_dep"] - tbl["first_dep"]) / 60
    hours = tbl["service_h"] if over_service_window else 24
    tbl["per_hour"] = (tbl["trips"] / hours).round(2)
    return (tbl.rename(columns={"route_short_name": "line"})
            .sort_values("trips", ascending=False).reset_index(drop=True))

# Re-run with the same filter as the first version (TRAIN_MODE = 3 → mode_codes=[3])
result = departures_per_hour(mode_codes=[3])
print("Lines found:", len(result))
print(result)

Lines found: 0
Empty DataFrame
Columns: [line, trips, per_hour]
Index: []


No crash this time — but look at what came back: an empty table. Zero lines. The code runs cleanly and confidently answers… nothing. Code that runs is not the same as code that's right, and an empty answer is the loudest possible version of that.

## 3. Read the code's logic — why is the table empty?

An empty result almost always means a filter matched nothing. The filter here keeps only mode code 3 — passed in as `mode_codes=[3]`, which becomes `route_type` equals 3 inside the function. Rather than assume what that code means, we ask the assistant to write a quick look at which codes the feed actually contains.

```markdown
The analysis came back empty, and I suspect the line filter is the cause. Write a short pandas snippet that prints every distinct `route_type` value in the `routes` table, with a plain-English name for each mode and how many lines have it, so I can see which modes this feed really has.
```

Running that gives:

In [3]:
"""Which mode codes actually exist in this feed?"""
import pandas as pd

routes = pd.read_csv(f"{DATA_DIR}/routes.txt", dtype=str)
routes["route_type"] = routes["route_type"].astype(int)

# GTFS mode codes (the standard list)
GTFS_MODES = {0: "tram", 1: "metro", 2: "rail", 3: "bus", 4: "ferry", 7: "funicular"}

for code, n in routes["route_type"].value_counts().sort_index().items():
    print(f"route_type {code} ({GTFS_MODES.get(code, 'other')}): {n} lines")

route_type 1 (metro): 4 lines
route_type 2 (rail): 14 lines
route_type 7 (funicular): 3 lines


There it is. The feed holds only mode codes 1, 2 and 7 — metro, rail and funicular. In GTFS, code 3 is *bus*, and FGC runs none. We passed it in as `mode_codes=[3]`, which inside the function keeps only rows where `route_type` equals 3 — a mode this feed doesn't have, so nothing survived. **Finding 1: the line filter keeps the wrong mode — the question asked about the whole network, the code kept only buses.**

## 4. Fix the filter and re-run

The question was about the whole FGC network, so the filter should keep all three real modes, not one. Because the corrected code made the filter a parameter, this is a one-line change.

```markdown
This feed has metro (1), rail (2) and funicular (7), and no buses. Re-run the analysis keeping all of the network's real modes instead of only one, and show the table of departures per hour by line.
```

Now the analysis returns real numbers:

In [4]:
# Keep the network's real modes: metro (1) + rail (2) + funicular (7)
result = departures_per_hour(mode_codes=[1, 2, 7])
print(f"{len(result)} lines, {int(result['trips'].sum())} weekday trips\n")
print(result[["line", "trips", "per_hour"]].to_string(index=False))

20 lines, 2041 weekday trips

line  trips  per_hour
  FV    392     16.33
  L7    277     11.54
 L12    237      9.88
  S2    235      9.79
  S1    235      9.79
  L6    177      7.38
  L8     99      4.12
  S8     80      3.33
  MM     59      2.46
  R5     59      2.46
  R6     59      2.46
  S3     47      1.96
  S4     40      1.67
 RL1     12      0.50
 RL2     10      0.42
 R60      6      0.25
 R50      6      0.25
  S9      6      0.25
 R63      3      0.12
 R53      2      0.08


Twenty lines and about 2,041 weekday trips, from the busy funicular at the top down to the small Lleida branches. The numbers look plausible at a glance — which is precisely the moment a careful analyst slows down instead of speeding up.

## 5. Spot-check one line by hand

Before trusting any per-hour figure, confirm the counts underneath them. The habit is to take one line you can reason about and count it a completely different way, then compare. L6 — the Barcelona–Sarrià line most locals know — reported 177 trips; let's count it straight from the trips table.

```markdown
Write a short pandas snippet that counts, directly from the trips table, how many trips line L6 runs on the same weekday — without calling the frequency function — so I can compare it against the 177 the table reported.
```

The independent count:

In [5]:
"""Spot-check: count L6's weekday trips straight from the trips table."""
import pandas as pd

routes = pd.read_csv(f"{DATA_DIR}/routes.txt", dtype=str)
trips = pd.read_csv(f"{DATA_DIR}/trips.txt", dtype=str)
calendar = pd.read_csv(f"{DATA_DIR}/calendar_dates.txt", dtype=str)

l6_id = routes.loc[routes["route_short_name"] == "L6", "route_id"].iloc[0]
running = calendar[(calendar["date"] == SERVICE_DAY)
                   & (calendar["exception_type"] == "1")]
l6_trips = trips[(trips["route_id"] == l6_id)
                 & (trips["service_id"].isin(running["service_id"]))]

print("L6 trips counted by hand:", len(l6_trips))
print("L6 trips in the table:   ", int(result.loc[result["line"] == "L6", "trips"].iloc[0]))

L6 trips counted by hand: 177
L6 trips in the table:    177


177 both ways. The counting underneath the table is sound — so if a frequency figure is wrong, the mistake is in what we *do* with the counts, not in the counts themselves. That narrows where to look next.

## 6. Sanity-check the numbers against the real world

Now read the frequencies the way a rider would. One line stands out: the Montserrat rack railway, MM, shows about 2.5 departures an hour — one roughly every 24 minutes — as a whole-day average. That only makes sense if the line runs all day. Let's check the hours it actually operates.

```markdown
For the Montserrat line (MM), write a snippet that prints its first and last departure of the day and its total number of trips, so I can see the window of hours it really operates.
```

Which shows:

In [6]:
"""How many hours does the Montserrat line actually operate?"""
def clock(mins):
    return f"{int(mins) // 60 % 24:02d}:{int(mins) % 60:02d}"

mm = result.loc[result["line"] == "MM"].iloc[0]
print(f"MM: {int(mm['trips'])} trips, "
      f"first {clock(mm['first_dep'])}, last {clock(mm['last_dep'])}")
print(f"per-hour as computed (over a 24-hour day): {mm['per_hour']} "
      f"-> about one train every {round(60 / mm['per_hour'])} minutes")

MM: 59 trips, first 07:48, last 19:35
per-hour as computed (over a 24-hour day): 2.46 -> about one train every 24 minutes


MM runs from about a quarter to eight in the morning to half past seven at night — under twelve hours. Yet its per-hour figure was found by spreading 59 trips across a full 24. Averaging service over hours when the line is closed drags every line's frequency down. **Finding 2: the per-hour figure divides by 24 hours, not by the hours each line is actually open.**

## 7. Ask the assistant to write sanity-check code

Two things are worth pinning down automatically rather than by eye: that no line quietly vanished when we changed the filter, and that the denominator really is the culprit. We describe both in plain language and let the assistant write the checks.

```markdown
Write two checks in pandas. First, a completeness check that asserts the corrected table has no missing or non-positive trip counts and that the total across all lines is over a thousand. Second, a cross-check that computes each line's departures per hour a second way — dividing by the hours between its first and last departure instead of by 24 — and flags the regularly-running lines (at least 20 trips a day) where the two methods disagree by more than 20 percent.
```

The checks it writes, and their output:

In [7]:
"""Two checks: completeness, and a cross-check of the per-hour method."""
import pandas as pd

# --- Check A: completeness (we expect this to PASS) ---
table = departures_per_hour(mode_codes=[1, 2, 7])
assert table["trips"].notna().all(),   "a line has a missing trip count"
assert (table["trips"] > 0).all(),     "a line has a non-positive trip count"
assert int(table["trips"].sum()) > 1000, "far fewer trips than expected"
print("Completeness check PASSED —",
      int(table["trips"].sum()), "trips across", len(table), "lines\n")

# --- Check B: compute per-hour a second way and compare ---
by_24     = departures_per_hour([1, 2, 7], over_service_window=False)
by_window = departures_per_hour([1, 2, 7], over_service_window=True)
cross = by_24[["line", "trips", "per_hour"]].merge(
    by_window[["line", "per_hour"]], on="line",
    suffixes=("_over_24h", "_over_window"))
cross = cross[cross["trips"] >= 20]                      # regularly-running lines
cross["pct_gap"] = ((cross["per_hour_over_window"] - cross["per_hour_over_24h"])
                    / cross["per_hour_over_24h"] * 100).round(0)
flagged = cross[cross["pct_gap"] > 20].sort_values("pct_gap", ascending=False)
print("Lines where the two methods disagree by more than 20%:")
print(flagged.to_string(index=False))

Completeness check PASSED — 2041 trips across 20 lines



Lines where the two methods disagree by more than 20%:
line  trips  per_hour_over_24h  per_hour_over_window  pct_gap
  MM     59               2.46                  5.01    104.0
  S3     47               1.96                  2.92     49.0
  R5     59               2.46                  3.27     33.0
  L7    277              11.54                 15.28     32.0
  S4     40               1.67                  2.21     32.0
 L12    237               9.88                 12.74     29.0
  R6     59               2.46                  3.18     29.0
  L8     99               4.12                  5.28     28.0
  L6    177               7.38                  9.34     27.0
  S2    235               9.79                 12.37     26.0
  S8     80               3.33                  4.21     26.0
  FV    392              16.33                 20.03     23.0
  S1    235               9.79                 11.87     21.0


The completeness check passes — nothing dropped, 2,041 trips in all. The cross-check is decisive: *every* regularly-running line disagrees by more than a quarter, and Montserrat by more than double. Computing the same quantity two ways and watching them diverge is how a hunch about the denominator becomes evidence. That confirms Finding 2.

## 8. Ask the assistant to review its own code

Human reading and automated checks each found something. Now bring in a third angle: hand the original code and the question back to the assistant and ask it to *review*, not rewrite. A reviewer prompt looks at the same work from the outside, and often names problems the author pass glossed over.

```markdown
Here is the original analysis you wrote and the question it was meant to answer: "on a representative weekday, what is the average number of departures per hour for each line, across the whole FGC network?" Review the code as a critic. Check that the line filter, the counting, and the per-hour calculation each match that question, and list any mismatches you find.
```

Its review:

```markdown
Two mismatches against the question:

1. **Line filter.** The call passes `mode_codes=[3]`, which inside the function keeps only lines whose `route_type` is 3. The question is about the whole network, but in GTFS code 3 is *bus* — and this feed has none, so the filter excludes everything. It should keep the modes the network actually runs (metro, rail, funicular), or not filter by mode at all.

2. **Per-hour denominator.** Departures are divided by 24. The question asks how often a line runs, and no line runs 24 hours a day — so dividing by 24 understates every line's frequency, and understates it most for lines with short operating windows. Divide instead by the hours between the line's first and last departure.

The counting itself (trips per line, on the selected weekday) looks correct.
```

Looking at the same code from the outside, the reviewer independently named both problems — the wrong mode filter and the 24-hour denominator — and, usefully, confirmed the counting is fine. Three separate angles, the same two findings.

## 9. Ask for a verification plan, then run one check

One loose end remains: we *believe* the corrected per-hour numbers, but belief isn't a check. Rather than invent tests, we ask the assistant to propose a short verification plan, then read the plan and run only the part that's actually feasible.

```markdown
Given the question, the corrected code, and the output table, propose two or three fast, concrete ways I could check that a specific line's departures-per-hour figure is right. Keep them to checks I can run against the data I already have.
```

Its proposed plan:

```markdown
1. Recompute one line from scratch: take that line's trips on the chosen day, read their departure times straight from stop_times, and divide by the span between the first and last departure. It should match the table.
2. Reconcile the total: the sum of every line's trips should equal the number of weekday trips in the trips table (nothing double-counted, nothing dropped).
3. Compare one line's figure against FGC's published timetable for that day.
```

Reading the plan first pays off: check 3 needs the public timetable, which we don't have offline, so we set it aside. Check 1 is exactly the independent recompute we want. We run it on L6:

In [8]:
"""Verification check: recompute L6's frequency from raw times over its own window."""
def clock(mins):
    return f"{int(mins) // 60 % 24:02d}:{int(mins) % 60:02d}"

l6 = departures_per_hour([1, 2, 7], over_service_window=True)
l6 = l6.loc[l6["line"] == "L6"].iloc[0]
print(f"L6 runs {clock(l6['first_dep'])}–{clock(l6['last_dep'])} (next day) "
      f"= {round(l6['service_h'], 1)} hours of service")
print(f"{int(l6['trips'])} departures / {round(l6['service_h'], 1)} h "
      f"= {l6['per_hour']} per hour  "
      f"(a train about every {round(60 / l6['per_hour'], 1)} minutes)")

L6 runs 05:10–00:07 (next day) = 19.0 hours of service
177 departures / 19.0 h = 9.34 per hour  (a train about every 6.4 minutes)


Recomputed straight from the raw departure times over L6's real service window, we get 9.3 departures an hour — a train roughly every six or seven minutes — matching the corrected table. Reading the plan before running it saved us from spending time on the one check we couldn't actually do.

## 10. Read the answer — the audit summary

Three layers of checking, run over a single analysis, turned up two logic errors the code itself never complained about — plus one plain crash on the way in.

| Finding | How we caught it | Result | Fix |
|---|---|---|---|
| Departure times past "24:00" crashed the time parsing | Ran the code | `ValueError` on `24:01:00` | Read times as minutes since midnight, not as a 24-hour clock |
| Line filter passed `mode_codes=[3]` (mode code 3 → bus), which FGC doesn't run | Read the code's logic; confirmed by the review | Empty table — the whole network excluded | Keep the real modes: metro (1), rail (2), funicular (7) |
| Departures-per-hour divided by 24, not by hours operated | Sanity-checked a number; confirmed by cross-check code and the review | Every line's frequency understated — Montserrat by half | Divide by each line's own window, first departure to last |

**Back to the question.** On a normal weekday, FGC runs 20 lines and about 2,041 scheduled trips. Once frequency is measured over the hours each line is actually open, the busiest lines land around 12 to 20 departures an hour, and L6 comes out at about 9 an hour — a train roughly every six or seven minutes — not the once-every-eight-minutes the draft implied. And the network is emphatically not empty. None of these fixes required rebuilding the analysis from scratch; each came from a quick, targeted check on work the assistant had already done — which, as models get better at writing the code, is where most of the job now lives.